In [1]:
import os

# safely change directory to project root
if os.path.split(os.getcwd())[-1] == "research":
    os.chdir("../")

%pwd

'/Users/apple/Aakash/Projects/wine-quality-e2e-production'

In [2]:

from dataclasses import dataclass
from pathlib import Path
from src.wine_quality_project.constants import CONFIG_FILE_PATH, SCHEMA_FILE_PATH, PARAMS_FILE_PATH
@dataclass
class DataIngestionConfig:
    root_dir:Path
    source_URL:str
    local_data_file:Path
    unzip_dir:Path

In [3]:
from src.wine_quality_project.constants import *
from src.wine_quality_project.utils.common import read_yaml, create_directories


class ConfigrationManager:
    def __init__(self,
                 config_filepath= CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH
                 ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config=DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir

        )
        return data_ingestion_config

In [ ]:
import os
import urllib.request as request
from src.wine_quality_project import logger
import zipfile

In [5]:
# components - DataIngestion

class DataIngestion:
    def __init__(self, config:DataIngestionConfig):
        self.config = config
    
    #DOWNLOAD FILE
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers= request.urlretrieve(
                url = self.config.source_URL,
                filename=self.config.local_data_file
            )

            logger.info(f"{filename} download! with following information : \n{headers}" )
        else:
            logger.info(f"file aready exits of size!")
    
    #EXTRACT FILE
    def extract_zip_file(self):
        """
        zip_file_path = str
        extract zip file into data dir
        Function return none
        """

        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)


In [6]:
try:
    config = ConfigrationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e


[2026-05-07 12:17:57,029: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-05-07 12:17:57,031: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-07 12:17:57,032: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-05-07 12:17:57,033: INFO: common: created directory at: artifacts]
[2026-05-07 12:17:57,034: INFO: common: created directory at: artifacts/data_ingestion]
[2026-05-07 12:17:58,455: INFO: 1006494956: artifacts/data_ingestion/data.zip download! with following information : 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 7624:2904A5:3E2230:A09A9B:69FC359D
Accept-Ranges: bytes
Date: